# Chapter 26 Guided To-Do Practice Labs

**Book Reference:** Chapter 26 (Part V Capstone), Section 26.5 (Guided To-Do Practice)  
Run **Step 0** to activate `sharukh-tutor`, implement all 6 production ML workflow components (custom transformer, mixed preprocessor, Stratified/Group/Temporal CV, and `joblib` serialization), and pass each **class or function name (without parentheses!)** to `tutor.check(...)`. Use `tutor.hint("ch26_todo_XX", level=1..3)` whenever you want a progressive nudge.

In [ ]:
# @title Step 0: Click Play (▶) to Activate Interactive Hints & Auto-Checker
!pip install -q sharukh-tutor
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import (
    GroupKFold,
    StratifiedKFold,
    TimeSeriesSplit,
    cross_val_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import sharukh_tutor as tutor

tutor.activate(chapter="ch26", notebook="guided_todos")


---
## To-Do 26.1 (Custom Transformer): `SentinelToNaNTransformer`

Build a Scikit-Learn compatible custom transformer class `SentinelToNaNTransformer(BaseEstimator, TransformerMixin)`:
- `__init__(self, sentinel=999)`: store `self.sentinel = sentinel` unmodified so `sklearn.base.clone()` works.
- `fit(self, X, y=None)`: return `self`.
- `transform(self, X)`: return a **copy** of `X` with every value equal to `self.sentinel` replaced by `np.nan` — supporting **both** `pd.DataFrame` and `np.ndarray` (including integer arrays) without mutating `X`.

In [ ]:
class SentinelToNaNTransformer(BaseEstimator, TransformerMixin):
    """Replace numeric sentinel codes (e.g., 999) with np.nan."""

    def __init__(self, sentinel=999):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        self.sentinel = sentinel
        ### END SOLUTION

    def fit(self, X, y=None):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        return self
        ### END SOLUTION

    def transform(self, X):
        # STUB: raise NotImplementedError("Replace with your solution")
        ### BEGIN SOLUTION
        if isinstance(X, pd.DataFrame):
            return X.copy().replace(self.sentinel, np.nan)
        arr = np.asarray(X, dtype=float).copy()
        arr[arr == self.sentinel] = np.nan
        return arr
        ### END SOLUTION


In [ ]:
tutor.check("ch26_todo_01", SentinelToNaNTransformer)


---
## To-Do 26.2 (Mixed-Type Preprocessor): `build_mixed_preprocessor(num_cols, cat_cols)`

Implement `build_mixed_preprocessor(num_cols, cat_cols)` returning a `ColumnTransformer` that applies:
- On `num_cols`: a `Pipeline` with `SimpleImputer(strategy="median")` followed by `StandardScaler()`.
- On `cat_cols`: a `Pipeline` with `SimpleImputer(strategy="most_frequent")` followed by `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`.

In [ ]:
def build_mixed_preprocessor(num_cols, cat_cols):
    """Return a ColumnTransformer for numeric + categorical features."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(
            handle_unknown="ignore", sparse_output=False
        )),
    ])
    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols),
    ])
    ### END SOLUTION


In [ ]:
tutor.check("ch26_todo_02", build_mixed_preprocessor)


---
## To-Do 26.3 (Stratified K-Fold CV): `cross_validate_pipeline(pipe, X, y, cv=5, seed=42)`

Implement `cross_validate_pipeline(pipe, X, y, cv=5, seed=42)`:
- Build `StratifiedKFold(n_splits=cv, shuffle=True, random_state=seed)`.
- Run `cross_val_score(pipe, X, y, cv=skf, scoring="roc_auc")`.
- Return `{"mean_roc_auc": round4, "std_roc_auc": round4, "fold_scores": list_round4}`.

In [ ]:
def cross_validate_pipeline(pipe, X, y, cv=5, seed=42):
    """Evaluate a pipeline via StratifiedKFold ROC-AUC cross-validation."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    skf = StratifiedKFold(
        n_splits=cv, shuffle=True, random_state=seed
    )
    scores = cross_val_score(pipe, X, y, cv=skf, scoring="roc_auc")
    return {
        "mean_roc_auc": round(float(scores.mean()), 4),
        "std_roc_auc": round(float(scores.std()), 4),
        "fold_scores": [round(float(s), 4) for s in scores],
    }
    ### END SOLUTION


In [ ]:
tutor.check("ch26_todo_03", cross_validate_pipeline)


---
## To-Do 26.4 (Cluster-Aware CV): `group_cv_evaluation(pipe, X, y, groups, n_splits=5)`

Implement `group_cv_evaluation(pipe, X, y, groups, n_splits=5)`:
- Build `GroupKFold(n_splits=n_splits)`.
- Run `cross_val_score(pipe, X, y, groups=groups, cv=gkf, scoring="roc_auc")`.
- Return `{"mean_roc_auc": round4, "std_roc_auc": round4, "n_splits": int}`.

In [ ]:
def group_cv_evaluation(pipe, X, y, groups, n_splits=5):
    """Evaluate a pipeline via GroupKFold ROC-AUC without group leaks."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    gkf = GroupKFold(n_splits=n_splits)
    scores = cross_val_score(
        pipe, X, y, groups=groups, cv=gkf, scoring="roc_auc"
    )
    return {
        "mean_roc_auc": round(float(scores.mean()), 4),
        "std_roc_auc": round(float(scores.std()), 4),
        "n_splits": int(n_splits),
    }
    ### END SOLUTION


In [ ]:
tutor.check("ch26_todo_04", group_cv_evaluation)


---
## To-Do 26.5 (Temporal Forecast CV): `temporal_cv_forecast_rmse(pipe, X, y, n_splits=4)`

Implement `temporal_cv_forecast_rmse(pipe, X, y, n_splits=4)`:
- Build `TimeSeriesSplit(n_splits=n_splits)`.
- Run `cross_val_score(pipe, X, y, cv=tscv, scoring="neg_root_mean_squared_error")`.
- Negate the scores so RMSE values are **positive**, and return `{"mean_rmse": round4, "fold_rmses": list_round4}`.

In [ ]:
def temporal_cv_forecast_rmse(pipe, X, y, n_splits=4):
    """Evaluate chronological forecasting RMSE via TimeSeriesSplit."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    tscv = TimeSeriesSplit(n_splits=n_splits)
    rmses = -cross_val_score(
        pipe, X, y, cv=tscv, scoring="neg_root_mean_squared_error"
    )
    return {
        "mean_rmse": round(float(rmses.mean()), 4),
        "fold_rmses": [round(float(r), 4) for r in rmses],
    }
    ### END SOLUTION


In [ ]:
tutor.check("ch26_todo_05", temporal_cv_forecast_rmse)


---
## To-Do 26.6 (Pipeline Serialization): `save_and_reload_pipeline(pipe, model_path, X_sample)`

Implement `save_and_reload_pipeline(pipe, model_path, X_sample)`:
- Save the fitted `pipe` to `model_path` via `joblib.dump(pipe, model_path)`.
- Reload it via `joblib.load(model_path)`.
- Compare `pipe.predict(X_sample)` against `reloaded.predict(X_sample)`.
- Return `{"file_bytes": int, "predictions_match": bool, "reloaded_preds": preds}`.

In [ ]:
def save_and_reload_pipeline(pipe, model_path, X_sample):
    """Dump a fitted pipeline with joblib, reload, and verify preds."""
    # STUB: raise NotImplementedError("Replace with your solution")
    ### BEGIN SOLUTION
    joblib.dump(pipe, model_path)
    reloaded = joblib.load(model_path)
    orig_preds = pipe.predict(X_sample)
    reloaded_preds = reloaded.predict(X_sample)
    return {
        "file_bytes": int(Path(model_path).stat().st_size),
        "predictions_match": bool(
            np.allclose(orig_preds, reloaded_preds)
        ),
        "reloaded_preds": reloaded_preds,
    }
    ### END SOLUTION


In [ ]:
tutor.check("ch26_todo_06", save_and_reload_pipeline)


---
## Annotated Debugging Post-Mortem (Solution Notebook Only)

- **To-Do 26.1 (Integer ndarray `NaN` Assignment & `clone()`):** Assigning `np.nan` into an integer NumPy array raises `ValueError: cannot convert float NaN to integer`. Always cast arrays with `np.asarray(X, dtype=float).copy()`, and store `self.sentinel = sentinel` unchanged in `__init__` so `sklearn.base.clone()` succeeds during cross-validation.
- **To-Do 26.2 (`handle_unknown="ignore"` & `sparse_output=False`):** Leaving `OneHotEncoder` at its default `handle_unknown="error"` crashes production scoring the first time an unseen district or clinic tier appears in test data. Setting `sparse_output=False` also ensures downstream tree models like `HistGradientBoostingClassifier` receive a clean dense array.
- **To-Do 26.5 (Sign Flip on Scikit-Learn Loss Scorers):** Scikit-Learn follows a "higher-is-better" convention across all `cross_val_score` and `GridSearchCV` metrics, so `neg_root_mean_squared_error` returns negative numbers. Always negate (`-scores`) before reporting RMSE to clinicians or reviewers.

In [ ]:
tutor.summary()
